# 02. 컬럼명 마스킹 데이터 생성

모델이 컬럼명·테이블명에 기대지 않는지 검증하기 위해, 테이블/컬럼 이름을 salt 해시로 바꾼
`<db>_mask` 버전을 만든다. 정답 파일도 같은 규칙으로 변환하고, 원본↔마스킹 매핑(`mask_mapping.json`)을 저장한다.

# BIRD Data Masking
컬럼명, 파일명(테이블명)을 해시값으로 익명화하고, ground_truth.json도 동일하게 치환합니다.
같은 컬럼명이라도 테이블이 다르면 서로 다른 해시값을 갖습니다.

In [ ]:
import hashlib
import secrets
import json
import csv as _csv
from io import StringIO
from pathlib import Path
import pandas as pd

In [ ]:
db_name = 'california_schools'
bird_root = Path(f'../workspace/dataset_mini/bird_data_tables/{db_name}')
output_root = Path(f'../workspace/dataset_mini/bird_data_tables/{db_name}_mask')
output_root.mkdir(parents=True, exist_ok=True)

## 1. Salt 생성 및 매핑 구성
- 테이블명: `hash(salt + table)`
- 컬럼명: `hash(salt + table + col)` → 같은 컬럼명도 테이블마다 다른 해시

In [ ]:
salt = secrets.token_hex(8)  # 실험마다 고정 seed
print(f'salt: {salt}')

csv_files = sorted(bird_root.glob('*.csv'))
print(f'CSV 파일 목록: {[f.name for f in csv_files]}')

In [ ]:
def make_hash(salt: str, *parts: str) -> str:
    key = salt + ''.join(parts)
    return hashlib.md5(key.encode()).hexdigest()[:6]

# 테이블명 매핑: hash(salt + table)
all_tables = [f.stem for f in csv_files]
table_mapping = {tbl: f'tbl_{make_hash(salt, tbl)}' for tbl in all_tables}
print('테이블명 매핑:')
for k, v in table_mapping.items():
    print(f'  {k} -> {v}')

In [ ]:
# 컬럼명 매핑: hash(salt + table + col) — 테이블마다 독립적인 해시
# col_mapping[table][original_col] = masked_col
col_mapping: dict[str, dict[str, str]] = {}

for f in csv_files:
    tbl = f.stem
    df_header = pd.read_csv(f, nrows=0)
    col_mapping[tbl] = {
        col: f'col_{make_hash(salt, tbl, col)}'
        for col in df_header.columns
    }

print('컬럼명 매핑 (테이블별):')
for tbl, mapping in col_mapping.items():
    print(f'\n[{tbl}]')
    for orig, masked in mapping.items():
        print(f'  {orig} -> {masked}')

## 2. CSV 익명화 및 저장

In [ ]:
for f in csv_files:
    tbl = f.stem
    masked_table_name = table_mapping[tbl]
    out_path = output_root / f'{masked_table_name}.csv'

    with open(f, 'rb') as infile:
        raw = infile.read()

    # 첫 줄(헤더)과 나머지 데이터 분리
    first_nl = raw.index(b'\n')
    header_bytes = raw[:first_nl]
    data_bytes   = raw[first_nl:]   # '\n' 포함

    # 헤더 파싱 (quoted 컬럼명 처리)
    orig_cols = next(_csv.reader([header_bytes.decode('utf-8-sig')]))
    masked_cols = [col_mapping[tbl].get(c, c) for c in orig_cols]

    # 마스킹된 헤더 생성
    buf = StringIO()
    _csv.writer(buf).writerow(masked_cols)
    masked_header = buf.getvalue().rstrip('\r\n').encode('utf-8')

    with open(out_path, 'wb') as outfile:
        outfile.write(masked_header)
        outfile.write(data_bytes)   # 데이터 행은 원본 바이트 그대로

    row_count = raw.count(b'\n') - 1  # 헤더 제외
    print(f'{f.name} -> {out_path.name}  (rows={row_count}, cols={len(orig_cols)})')

## 3. ground_truth.json 익명화
`"table.column"` 참조를 테이블별 해시값으로 치환합니다.

In [ ]:
gt_path = bird_root / 'ground_truth.json'
with open(gt_path, 'r', encoding='utf-8') as f:
    ground_truth = json.load(f)

print('원본 ground_truth.json:')
print(json.dumps(ground_truth, indent=2, ensure_ascii=False))

In [ ]:
def mask_ref(ref: str) -> str:
    """'table.column' -> 'masked_table.masked_col' (테이블별 독립 해시)"""
    parts = ref.split('.', 1)
    if len(parts) == 2:
        tbl, col = parts
        masked_tbl = table_mapping.get(tbl, tbl)
        masked_col = col_mapping.get(tbl, {}).get(col, col)
        return f'{masked_tbl}.{masked_col}'
    return ref

masked_gt = [[mask_ref(ref) for ref in pair] for pair in ground_truth]

print('익명화된 ground_truth:')
print(json.dumps(masked_gt, indent=2, ensure_ascii=False))

In [ ]:
out_gt_path = output_root / 'ground_truth.json'
with open(out_gt_path, 'w', encoding='utf-8') as f:
    json.dump(masked_gt, f, indent=2, ensure_ascii=False)
print(f'저장 완료: {out_gt_path}')

## 4. 매핑 테이블 저장
재현 가능하도록 salt와 매핑 정보를 `mask_mapping.json`으로 저장합니다.

In [ ]:
mapping_record = {
    'salt': salt,
    'table_mapping': table_mapping,
    'col_mapping': col_mapping,
}
mapping_path = output_root / 'mask_mapping.json'
with open(mapping_path, 'w', encoding='utf-8') as f:
    json.dump(mapping_record, f, indent=2, ensure_ascii=False)
print(f'매핑 저장 완료: {mapping_path}')

In [ ]:
print('=== 최종 결과 ===')
for p in sorted(output_root.iterdir()):
    print(f'  {p.name}')